<a href="https://colab.research.google.com/github/Fatima-05/FlyRank-ML/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fatima-05/FlyRank-ML/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

Lane: Refresh / Content Opportunity Scoring.

Method: random forest classifier used as a ranker (sort by positive-class probability).
Also train a decision tree as a simpler reference.

Why it fits: the job is prioritising pages for review, not only global accuracy.
Tree models handle mixed numeric signals, need little scaling, and give a usable score for a ranked queue.

In [5]:
print("Method: RandomForestClassifier + DecisionTreeClassifier")
print("Use: rank pages by P(is_declining) for refresh opportunity scoring")

Method: RandomForestClassifier + DecisionTreeClassifier
Use: rank pages by P(is_declining) for refresh opportunity scoring


## 2. Split design

Split: GroupShuffleSplit on client_id (25% test, random_state=42).

Why honest for this question: pages from the same client can share style and tracking patterns.
Holding out whole clients reduces train/test leakage from repeated client structure.
This is not a full time-series seal, but it is stronger than a plain random row split.

In [6]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

if not Path("data/raw/content_refresh_anonymized.csv").exists():
    if not Path("/content/FlyRank-ML").exists():
        !git clone https://github.com/Fatima-05/FlyRank-ML.git /content/FlyRank-ML
    os.chdir("/content/FlyRank-ML")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)

feature_cols = [c for c in [
    "impressions_90d", "clicks_90d", "ctr", "content_age_days",
    "word_count", "avg_position", "search_volume"
] if c in df.columns]

X = df[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
y = df["is_declining"]
groups = df["client_id"] if "client_id" in df.columns else pd.Series(np.arange(len(df)))

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

print("Features:", feature_cols)
print("Train:", len(X_tr), "Test:", len(X_te))
print("Test base rate (declining):", round(y_te.mean(), 3))

Features: ['impressions_90d', 'clicks_90d', 'ctr', 'content_age_days', 'word_count', 'avg_position', 'search_volume']
Train: 22885 Test: 7115
Test base rate (declining): 0.517


## 3. Train + compare vs my baseline

Same data, same split, same metric: Precision@50.
Baseline is a hand-written score using non-label signals only.

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

def baseline_score(row):
    score = 0
    imp = row.get("impressions_90d", 0) or 0
    age = row.get("content_age_days", 0) or 0
    ctr = row.get("ctr", None)
    pos = row.get("avg_position", None)
    words = row.get("word_count", 0) or 0
    if imp >= 5000: score += 30
    elif imp >= 2000: score += 20
    elif imp >= 500: score += 10
    if age >= 365: score += 25
    elif age >= 180: score += 15
    elif age >= 90: score += 5
    if ctr is not None and imp >= 500:
        if ctr < 0.01: score += 20
        elif ctr < 0.02: score += 10
    if pos is not None:
        if 4 <= pos <= 15: score += 15
        elif 15 < pos <= 30: score += 5
    if words > 0 and words < 400: score += 5
    return score

def precision_at_k(frame, score_col, k=50):
    top = frame.sort_values(score_col, ascending=False).head(k)
    return float(top["is_declining"].mean())

test = df.iloc[test_idx].copy()
test["baseline_score"] = test.apply(baseline_score, axis=1)

rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
dt = DecisionTreeClassifier(max_depth=6, random_state=42)
rf.fit(X_tr, y_tr)
dt.fit(X_tr, y_tr)

test["rf_score"] = rf.predict_proba(X_te)[:, 1]
test["dt_score"] = dt.predict_proba(X_te)[:, 1]

results = pd.DataFrame({
    "model": ["baseline_rules", "decision_tree", "random_forest"],
    "precision_at_50": [
        precision_at_k(test, "baseline_score"),
        precision_at_k(test, "dt_score"),
        precision_at_k(test, "rf_score"),
    ],
})
print(results)
results

            model  precision_at_50
0  baseline_rules             0.50
1   decision_tree             0.42
2   random_forest             0.62


,model,precision_at_50
0,baseline_rules,0.50
1,decision_tree,0.42
2,random_forest,0.62


## 4. Errors and interpretation

Look at top-ranked misses and what the forest leans on.
This is short error analysis, not a claim of causal refresh impact.

In [8]:
# top 20 by RF score: how many are actually declining?
top20 = test.sort_values("rf_score", ascending=False).head(20).copy()
print("Top20 declining rate:", round(top20["is_declining"].mean(), 3))
print("Top20 false priorities (is_declining==0):", int((top20["is_declining"] == 0).sum()))

# feature importance
imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nFeature importance:")
print(imp)

# a few false priorities for inspection (no client names)
cols_show = [c for c in ["content_id", "rf_score", "is_declining", "impressions_90d", "ctr", "content_age_days", "avg_position"] if c in top20.columns]
false_pos = top20[top20["is_declining"] == 0][cols_show].head(5)
print("\nExample false priorities in top20:")
false_pos

Top20 declining rate: 0.6
Top20 false priorities (is_declining==0): 8

Feature importance:
impressions_90d     0.307543
avg_position        0.236693
content_age_days    0.180256
word_count          0.129798
clicks_90d          0.063301
ctr                 0.055236
search_volume       0.027174
dtype: float64

Example false priorities in top20:


,content_id,rf_score,is_declining,impressions_90d,ctr,content_age_days,avg_position
15705,content_4dd569ee33c9,0.820898,0,361,0.0,275,14.8
18475,content_197a5b1ed096,0.820840,0,605,0.0,223,14.4
22042,content_2ba626fea4d6,0.820807,0,360,0.0,275,7.2
28718,content_ef6e7d7cfe15,0.818753,0,264,0.0,271,22.2
20736,content_41baf0722ad9,0.818490,0,3115,0.0,275,12.8


Interpretation notes:
- If importance ranks age / impressions / CTR highly, the model is leaning on the same family of clues as the baseline, but combined non-linearly.
- False priorities in the top20 are why the queue stays a reviewer aid.
- Measured comparison only; no causal traffic claim.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.